⚠️ **This notebook is meant to be pedagogical. By definition, it is done linearly and covers a lot of different things. Given time constraints, it does not focus on giving "the best model", but rather to explain what is at stakes when training a model and how to deal with parameter tuning and deployment.**

# Partie 1 : Theory of machine learning (~45 mn)

We will use the document [theoretical_basis_of_ml.pdf](./theoretical_basis_of_ml.pdf). It shows that all machine learning problem can be rewritten as an optimisation problem
$$
\underset{g \in\mathcal{G}}{\argmin}\space \frac{1}{n} \sum_{i=1}^{n}l(Y_i, g(X_i))
$$

There are many different ways to optimise the search of a best model. We try to give insights and intuitions of which one will likely perform better in different situations (cf. document).

#### Option 1 - Reduce the variance : Optimise the search "inside" $\mathcal{G}$

- **Change the optimizer** method to make convergence faster / more precise, takin ginto account the loss function
- **Change the loss** $l$ to a "more regular" or "more convex" function (e.g. logit / softmax instead of error rate) to make convergence easier  
- **Reformat features** to help convergence : for instance, discretizing continuous features may help by reducing the $\mathcal{X}$ space  
- **Add regularisation hyperparameter** (e.g. l1 - lasso regression, l2 - ridge regression) to "smoothen" the $\mathcal{G}$ space 
- Convexify the loss by **using bagging estimator**: 
The key idea is that if we have $C_1, ..., C_T$ classifiers with the same distribution of mean $m$ and variance $\sigma^2$, then we can consider the average $C=\frac{1}{T}\sum_{k=1}^{T}C_k$. We have :
    - $\mathbb{E}[C]=\frac{1}{T}\mathbb{E}[\sum_{k=1}^{T}C_k]=\frac{1}{T}\sum_{k=1}^{T}\mathbb{E}[C_k]=\frac{T}{T}m=m$
    - $\mathbb{V}[C]=\frac{1}{T^2}\mathbb{V}[\sum_{k=1}^{T^2}C_k]=\frac{1}{T^2}(\sum_{k=1}^{T}\mathbb{V}[C_k]+2 \sum_{1 \leq i < j\leq T} Cov(C_i, C_j))=\frac{\sigma^2}{T} + \frac{2}{T} \sum_{1 \leq i < j\leq T} Cov(C_i, C_j)$ so if the correlation of the $C_k$ is "low" (which can be achieved by randomly removing rows and columns in the original dataset, or by downsampling), the variance is greatly reduced. In the limit case where they are independant, the variance is divided by T while the bias stay the same. For this reason it is highly recommended to create a bagged estimator when the estimator is unstable.
- **Resampling** (down, up, SMOTE) an unbalanced dataset to help convergence. Notice that it changes the distribution, instead of estimating $\mathbb{E}[Y|X]$ we estimate $\mathbb{E}[Y_{resampled}|X]$, but $\mathbb{E}[\mathbb{E}[Y|X]]=\mathbb{E}[Y] \neq \mathbb{E}[Y_{resampled}]$ so all estimated probabilities are wrong. 
    - This is *not an issue if we want to rank* because the order is preserved. 
    - This is *a big issue if the exact level of probability matter*, for instance to evalate an insurance premium because we will over or underestimating the risk of claims would lead to a wrong premium.

#### Option 2-  Reduce the bias: Optimise the search globally
- **Transform the features** to modify $\mathcal{G}$: for instance, if you use powered features or create product between features with a linear model, you now allow to search in a polynomial space. Notice that it does NOT create information, and it is not useful for all algorithm : for a tree a monotonic transformation of a continuous feature will not help. 
- Explore new spaces $\mathcal{G}$ by **changing algorithm** (e.g. GLM, random forest, SVM, gradient boosted trees...)
- **Explore new spaces $\mathcal{G}$ by changing hyperparameter** of a given family of algorithm : for instance if you are using boosted trees with fixed tree depth, you can change the tree depth and retrain to see if this new class $\mathcal{G}$ offer better predictive performance. This process is reffered to as **finetuning hyperparameters**. ⚠️Beware:  
    - **Parametric algorithms** (e.g GLM) make assumptions about the underlying distribution (the relation between X and Y is linear), which makes them **more underfitting prone** and makes more important the work on the features spaces $\mathcal{X}$ to redefine a good underlying space $\mathcal{G}$.
    - On the opposite **non parametric algorithm** like tree based methods or neural network makes very little assumptions on the underlying distribution. It makes them much more prone to overfitting, but also able to fit very coomplex distribution and less biased in many situations. Regularisation is very important for these kind of algorithms.
-  Change $\mathcal{X}$ : **Add new features** to your dataset gives more information, hence the conditional expectation of Y given X is necessarily greater. It has often much more impact on the end result than changing the space $\mathcal{G}$. You can try for instance different aggregates on different time periods, look for new data sources...


#### Option 3 - Redefine the problem to make it easier

- **Change the loss function $l$** : Depending on the problem, you may have different optima with different loss functions. Some metrics tend to flattten quickly (e.g. AUC is insenstitive to the absolute value of probabilities if the order of instances remain unchanged : it tends to "plateau" quickly while the convergence is not fully finished). It is recommended to **use several evalution metrics** to detect these different optima.
- For classification problems, it can be useful to **finetune the threshold** because many metrics are very threshold sensitive (e.g. precision, recall, accuracy) and the default 50% threshold may be *very* inadequate for your problem. For instance for an imbalanced dataset with 1% fraud, a 5% fraud prediction may be very high regarding the rest of the data, and worth investigating.
- **Evaluate differently** : When data are time sensitive, the distribution can change over time which violates our hypothesis. It is strongly recommended to create a train / test split  not at random but "time based" (i.e. train on one year and test on the following year) to detect these distribution shift (e.g. claims cost increase due to inflation).
- **Change $\mathcal{Y}$** : A problem can be framed in several way, which impact the underlying modelling hypothesis. For instance, the risk to churn (terminate your contract) will be identified byy differetn features if we predict churn in 1 month (likely "hot data" like recent claims, call to the call center, bad NPS, people with recent preimums increase...) vs 1 year (likely "cold data" like population with "above the market" premiums).


💡**It is recommended to use an AI (Copilot, ChatGPT...) in all the notebook to get help with the syntax, understand code...**

# Partie 2 : Create baselines model (~2h)

## 📝 Exercise 1 : load and explore the dataset ``ec_dde_tpe_2019_2023.pkl``. This dataset contains data for house insurance claims costs. 

Instructions : Use a copilot prompt to explore the dataset. You can for instance:
- Load the dataset from the path data/01_raw/ec_dde_tpe.pkl
- Display the first 5 rows of the dataset
- Display the last 5 rows of the dataset
- Display the shape of the dataset
- Display the columns of the dataset
- Display the data types of the columns
- Display the number of missing values in each column
- quickly plot / give distribution / number of missing values of the variables

💡**Copilot prompt example :**
> My claims_data_mrh dataframe has the following keys
> CLE DATE_NAISSANCE PRIMES MOBILIER OBJETS_VALEUR ZONIER DATE_SURVENANCE DATE_DECLARATION NB_SINISTRES CHARGE_SINISTRES
> I want to plot the number of claims per year with a bar chart grouped by GARANTIE


#### 🤔 Take a step back and comment on the data quality, and some feature transformation that will be needed. We want to predict the "CHARGE_SINISTRES" variable. 

- We likely need to distinguish between DDE and TEMPETE!
- some variables have null values which are unrealistic ( MOBILIER,  OBJETS_VALEUR...)
- somes variables are very correlated
- It does not make sense to predict the number of claims, because we don't have the "no claims" data in our dataset
- Some features will likely not be available at inference time, so you will need to drop them from the model
- features to remove: CLE, DATE_NAISSANCE, DATE_DECLARATION, DATE_SURVENANCE, GARANTIE, YEAR_SURVENANCE
- feature to transform : DATE_NAISSANCE-> AGE ; CHARGE_SINISTRES -> COUT_UNITAIRE; CODE_POSTAL -> DEPARTMENT


💡 **For the sake of simplicity and to keep on time constraints, we will remove some problematic variables and not impute / deal with missing data or outliers for now.** 

## 📝 Exercise 2 - Prepare the features
Instructions: 
- create variable "AGE"
- create variable "DEPARTMENT"
- create variable "COUT_UNITAIRE" as a label
- create_variable "HAS_EXTENSION"
- plot an histogram of "COUT_UNITAIRE" to check the distribution

Try to explain why we are doing these transformations!

⚠️ **Beware**: `CODE_POSTAL` has missing values, and they are not necessarily all represented the same way. An exact string match on a couple of known "missing" values can silently miss some rows. Double-check that your `DEPARTMENT` column has no leftover non-numeric values before moving on.

## 📝Exercise 3 - Train a baseline model

We are modelling a variable with stricly positive value and huge tail value. Gamma regression is a good fit for such target variable.  

Instructions:
- drop unused features
- for categorical values:
    - impute categorical value with the most frequent value
    - one hot encode the categorical variables (ensure you let 1 level out to avoid perfect correlations of features)
- for numerical values:
    - impute with mean 

⚠️ **Beware**: `SimpleImputer` has a `missing_values` parameter. Not all columns necessarily use the same "missing" sentinel under the hood (e.g. `pd.NA` vs. `np.nan`) — check the dtype of each column rather than assuming one setting works for every pipeline, or an imputer might silently do nothing.

In [20]:
X_train_transformed.head()

,PRIMES,MOBILIER,ZONIER,AGE,DEPARTEMENT,HAS_EXTENSION,USAGE_AUTRE,USAGE_PROFESSIONNEL,USAGE_RESIDENCE_PRINCIPALE,USAGE_RESIDENCE_SECONDAIRE,...,SURFACE_HAB_[66.0-78.0[,SURFACE_HAB_[78.0-89.0[,SURFACE_HAB_[89.0-100.0[,SURFACE_AUTRE_[0.0-1.0[,SURFACE_AUTRE_[1.0-15.0[,FORMULE_CONFORT,FORMULE_STANDARD,FORMULE_TOUT_RISQUE,GARANTIE_DDE,GARANTIE_TEMPETE
12557,218.04,21841.30,5.0,58.0,83.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
3379,132.55,11023.14,4.0,63.0,38.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
3921,274.71,65524.96,2.0,35.0,64.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
1056,272.42,66761.28,2.0,58.0,50.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
1146,101.00,34384.44,2.0,35.0,64.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


💡 **For reproducibility, use the make_pipeline and the ColumnTransformer API**. You will thank me later in the notebook ;) Please rewrite previous cell if you did not.

#### 🤔Take a step back and discuss the results
- This is quite a bad model : the rmse is of the same order of magnitude than the mean, so it is a 100% error rate!
- The prediction on test is reasonnably close to the average, which shows good convergence. 

## 📝 Exercise 4 -  Train an xgboost regressor

- Introduction to the algorithm : https://speakerd.s3.amazonaws.com/presentations/5c6dab45648344208185d2b1ab4fdc95/XGBoost-Newest.pdf
- Instructions:
    - Train a XGBRegressor from the xgboost library
    - predict on test dataset
    - evaluate rmse


#### 💡 Hints on the model:
- ``n_estimators``: the number of trees to build. The more trees, the better the model, but also the longer it takes to train and th greter risk of overffitting.. A good proxy is to take n_estimators in [1/eta; 10/eta] to ensure convergence is finished
- the learning_rate ``eta`` control the convergence speed : witht the right amount of trees, we should get almost the same minimum, hopefeully reach after 1/eta iterations.
- ``max_depth`` : the complexity of the trees. The more complex the trees, the less bias but the more variance at ach step. A good proxy is to take max_depth in [4,12], unless there are very complex relationships between columns, and a lot of columns
- always put a ``seed`` to make result reproducible.
- ``colsample_bytree``: the fraction of features to use for each tree. A good proxy is to take colsample_bytree in [0.5, 1]. The more columns the more we need to reduce colsample by tree to make trees independant and reduce variance. For some rare case with sparse data (e.g. nLP with bag of words), we can use colsample_bytree <0.5 to get a better result.
- ``subsample``: the fraction of rows to use for each tree. A good proxy is to take subsample in [0.5, 1]. The more rows the more we need to reduce subsample to make trees independant and reduce variance. It is also very usefl when the class is imbalanced
- ``min_child_weight``: the minimum sum of weights of all observations required in a child. For L2 loss this is the number of observations in an node. For L1 loss this is the sum of the absolute values of the observations. A good proxy is to take min_child_weight in [1,10] unless the  class is imbalanced. If max_depth is small, this is "naturally" limited becasue there are more observations per node

#### 💡Hints on the fitting procedure:
- **Early stopping is absolutely mandatory**, else the model will train all estimators nomatter if it is overfitting or not. The number of trees should not be estimated, but fitted with early_stopping. 
If early stopping has not be triggered, retrain with a higher number of trees.
- for NA,we **could avoid imputing** and let xgboost impute NA as a category; it chooses the best path in the branch for na
- imputing by a numeric out of range value (e;g. 1 for all positive) may makes sense because we would interpret all -1 as NA and quantifi its specific impact
- for categorical features, ``TargetEncoding`` or ``OrdinalEncoding`` are much better because you keep readibility (no multiple columns due to one hot encoding), but it is also possible to let xgboost discretize automatically

#### Analyse the results

Instruction: 
- plot feature importance with shap. Create a function to pot the feature importance.
- plot shap values and some graphs (bar, heatmap, and all the margins with scatter plots). Analyse the results to check if they are conform to your thoughts. 



Example of analysis: 
- people living in the zone 1 of the ZONIER have much higher cost in average.
- some deparments have much higher risk
- having an extension increase the average cost...
- TEMPETE are much more costly (>5k€ gap) than "DEGATS DES EAUX)

These results are mostly intuitive. 

# Part 3 - Hyperparameter tuning with optuna

- It would be very tedious to search for many combination of hyperparameters (expand \mathcal{G}) , but we feel that it would increase the predictive power
- For this, we will use a library called "optuna"(https://optuna.org/) which is a hyperparameter optimization framework that automates the search for the best hyperparameters.
- The key idea is to use an objective function to be optimized (e.g. your model)
- Why optuna? It uses Sequential Model Based Optimization (SMBO) instead of grid search or random search. This means it builds a probabilistic model of the objective function and uses it to select the most promising hyperparameters to evaluate next.

## 📝 Exercise 6 - Finetune the hyerparameters
Instructions: 
- create an optuna study which search for ``max_depth``, ``subsample``, ``colsample_bytree``, ``gamma``, ``min_child_weight``
- optimize for rmse
- launch several runs - start small and increase if you have enough computing power

#### 💡 Hints: Take Copilot suggestions with a grain of salt and try to reduce the number of hyperparameters to be optimized because you don't have an unlimited "budget" of time and computing power.


#### 🤔 Take a step back and discuss the results
- Are the different runs similar? why? The variance is high (between 700 and 900), indicating some instable results. This calls for 1/ regularization and 2/ bagging
- are the best results "intuitive" regarding previous explanation? Do some parameters hit the limit of the grid? If yes, this indicates that me might need to increase the size of the search window? Plot the rmse by feature to check for stability?
 

# Part 4 - Tracking your experiments with mlflow

When you are experimenting, you can get lost very fast because you keep running *slightly different* version of **code**, on *slightly different* versions of **data**, with *slightly different* **parameters**. It is hard to keep track of everything, and once you're done experimenting you want to get back to the best model you managed to train.

Here comes **Mlflow**: according to its documentaiton, MLflow is "*A Tool for Managing the Machine Learning Lifecycle*". It is composed of serveral module, and the first one is [MLflow Tracking](https://mlflow.org/docs/latest/tracking/)., which is dedictated to keep track of your ml experimentation. This is the module we will explore in this section. 

#### 💡 Hints on mlflow
- Configure the mlflow setup with ``mlflow.set_tracking_uri(Path().cwd().parent / "mlruns")``. It will create a ``mlruns`` folder at the root of the project
- Add ``mlflow.log_param()``, ``mlflow.log_metric()``, ``mlflow.log_artifact()`` (to log data), ``mlflow.log_figure()`` and ``mlflow.log_model()`` functions where they are needed and that'sis.
- Be careful : for ``mlflow.log_artifact()``, you need to persist the data as a file locally first.
- Open a terminal (in jupyter lab > New window with a + > terminal) and  launch ``mlflow ui``. Then open http://127.0.0.1:5000/
- ⚠️ Depending on your installed mlflow version, the local filesystem backend (``./mlruns``) may require an explicit opt-in via an environment variable before ``mlflow.start_run()`` will work. If you hit an error mentioning the filesystem backend, check mlflow's documentation for the relevant setting.

## 📝 Exercise 7 - Tracking with mlflow
Instructions: 
- copy paste the same code as for the gamma regressor, and add mlflow where needed to keep track of everything you need to make it reproducible

Instructions: Do the same with the optuna code

# Part 5 - Deploying your machine learning pipeline as a custom mlflow model

This is nice, but now comes the real goal : deploying our model. it is unrealistic to ask our users to transform the dataset  before calling the model. There are very little chances we got the correct input and transformation. To fix this problem, let assume our user will sent us the following input:


Notice that the only thing the user has to do is to filter for unused columns. All transfomations will be made internally by our model.  

For this purpose, mlflow has a module called [MLflow Models](https://mlflow.org/docs/latest/traditional-ml/creating-custom-pyfunc/notebooks/override-predict#defining-our-custom-pythonmodel) which enable you to create custom "model". A model is a standardized format that mlflow can deploy to many environments. 

## 📝 Exercise 8: Create a custom model with all your transformations

Instructions: Fill the blanks in below code by adding necessary preprocessing steps to our custom model.

In [ ]:
import mlflow.pyfunc
from mlflow.models import infer_signature


class SkearnPipelineModel(mlflow.pyfunc.PythonModel):
    def __init__(self, column_transformer, model):
        self.model = model
        self.column_transformer = column_transformer

    def load_context(self, context):
        with open(context.artifacts["column_transformer_path"], "rb") as f:
            self.column_transformer = pickle.load(f)

        with open(context.artifacts["model_path"], "rb") as f:
            self.model = pickle.load(f)

    def predict(self, context, model_input, params=None):
        """
        Args:
        - model_input (pd.DataFrame): DataFrame containing columns 'a' and 'b'.
        - params (dict, optional): Dictionary containing optional parameter 'delta'.
        """
        # BEWARE: we do not want to put too many transformations in the model, because we will need to retrain the model for each preprocessing change
        # It would be better to have all these preprocesssing steps in a function...
        model_input["AGE"] = ...
        model_input["HAS_EXTENSION"] = ...

        # imputation method should be discussed for missing value, this is liley a bad method!
        model_input["DEPARTEMENT"] = ...

        features = model_input.drop(
            columns=[...
            ]
        )  # Drop unnecessary columns

        for col in features.select_dtypes(include=["object"]).columns:
            features[col] = features[col].astype("category")

        # Transform the input data using the column transformer
        transformed_input = ...
        # Make predictions using the trained model
        predictions = self.model.predict(...)
        return pd.DataFrame(
            predictions, index=model_input.index, columns=["pred_COUT_UNITAIRE"]
        )


Instructions: Now, we once again copy past the training code. Complete the block in the middle which logs our custom model instead of "only" the gamma_regressor.

In [ ]:
with mlflow.start_run(run_name="gamma_regression"):
    
    categorical_pipeline = make_pipeline(
        SimpleImputer(missing_values=pd.NA, strategy="most_frequent"),
        OneHotEncoder(sparse_output=False, handle_unknown="error"),
    )
    
    numeric_pipeline = make_pipeline(SimpleImputer(missing_values=pd.NA, strategy="mean"))
    
    column_transformer = ColumnTransformer(
        (
            ("numerical", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ),
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    column_transformer.fit(X_train)

    # log the artifact for further reuse. First persist on disk. 
    with open("column_transformer.pkl", "wb") as f:
        pickle.dump(column_transformer, f)
    mlflow.log_artifact("column_transformer.pkl", artifact_path="preprocessing")
    
    X_train_transformed = column_transformer.transform(X_train)
    X_test_transformed = column_transformer.transform(X_test)

    # log_params  for reproducibility
    mlflow.log_param("model_type", "GammaRegressor")
    mlflow.log_param("solver", solver:="lbfgs")
    mlflow.log_param("max_iter", max_iter:=10_000)
    mlflow.log_param("fit_intercept", fit_intercept:=True)


    gamma_model = GammaRegressor(fit_intercept=fit_intercept, solver=solver, max_iter=max_iter)
    gamma_model.fit(X_train_transformed, y_train)



    
    # COMPLETE FROM HERE -------------------------------------------------------------
    
    # Save the model in custom format
    # we can optionally define param: params={"method": "predict_above_threshodl"}
    with open("gamma_model.pkl", "wb") as f:
        pickle.dump(gamma_model, f)


    signature_data = claims_data_mrh[
        [# 'CLE', unused
         'DATE_NAISSANCE', # used for AGE
         'CODE_POSTAL', # used for DEPARTMENT
         'PRIMES',
         'MOBILIER',
         # 'OBJETS_VALEUR', unused
         'ZONIER',
         'USAGE',
         'NOMBRE_PIECES',
         'NOMBRE_ETAGES',
         'SURFACE_HAB',
         'EXTENSION_HAB', # used for HAS_EXTENSION
         'SURFACE_AUTRE',
         'FORMULE',
         'DATE_SURVENANCE', # used for AGE
         'DATE_DECLARATION', # unsued
         'GARANTIE',
         # 'NB_SINISTRES',  # labels
         # 'CHARGE_SINISTRES' # label
        ]
    ]# Drop unnecessary columns

    
    signature = infer_signature(...)

    mlflow.pyfunc.log_model(
        ...
    )

    
        # END COMPLETE -------------------------------------------------------------

    
    coefficients = pd.DataFrame(
        {"Feature": X_train_transformed.columns, "Coefficient": gamma_model.coef_}
    ).sort_values(by="Coefficient", ascending=False)

    # log coefficients asa artifacts for analysis
    coefficients.to_csv("coefficients_gamma.csv", index=False)
    mlflow.log_artifact("coefficients_gamma.csv", artifact_path="analysis")

    # Make predictions
    y_pred_train = pd.DataFrame(
        gamma_model.predict(X_train_transformed),
        index=X_train.index,
        columns=["pred_COUT_UNITAIRE"],
    )
    y_pred_test = pd.DataFrame(
        gamma_model.predict(X_test_transformed),
        index=X_test.index,
        columns=["pred_COUT_UNITAIRE"],
    )

    print(y_train.mean(), y_pred_train.mean())
    print(y_test.mean(), y_pred_test.mean())

    # Evaluate the model
    mae = mean_absolute_error(y_test, y_pred_test)
    rmse = root_mean_squared_error(y_test, y_pred_test)

    # log_metrics for analysis
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    
    RUN_ID = mlflow.active_run().info.run_id # keep the run id for next cell


#### 📝 Exercise 9:  The moment of truth : predict on new data!

Instructions : 
- read claims from 2024 "ec_dde_tpe_2024.pkl"
- load the model from mlflow
- predict on enw data

In [68]:
claims_data_mrh_2024 = pd.read_pickle(
    Path().cwd().parent / "data" / "01_raw" / "ec_dde_tpe_2024.pkl"
)
instances_2024=claims_data_mrh_2024.drop(['CHARGE_SINISTRES', 'CLE', 'OBJETS_VALEUR', 'NB_SINISTRES'], axis=1)

prod_model = mlflow.pyfunc.load_model(f"runs:/{RUN_ID}/custom_model")

y_preds_validation=prod_model.predict(instances_2024)

#### Small conclusion : does it perform well? 

In [70]:
(claims_data_mrh_2024["CHARGE_SINISTRES"]/claims_data_mrh_2024["NB_SINISTRES"]).mean(),  y_preds_validation.mean()

(np.float64(1113.693045941807),
 pred_COUT_UNITAIRE    1664.570175
 dtype: float64)

#### ☠️ The model **perform very poorly on validation data.** But why? The answer is that there is a huge variability for TEMPETE risk depending on year. Our method of splitting between train and test was purposedly bad, hoped someone spotted it before we got to this extent ;) The correct approach would have been to split on a **time based maneer** to detect this overfitting much earlier!

# Going further

There are many (many many!) ways to improve the model : 
- do much **more feature engineering** instead of dropping features
- try **other models** (e.g. SVM, LightGBM...) 
- try **bagging** approaches
- try getting **open data** to enrich our dataset (e.g. other risk based on CODE_POSTAL)
- do **a separate modelling between DDE and  TEMPETE** which are not the same type for risk
- **adjust label for inflation**
- **adjust label for climate risk** increase
- change metrics (e.g. discretize the label and use a classification approach with tuned threshold for multiclass precision and recall...)  
- ...

We could also **serve the model as an API with "mlflow serve"** but this is another story.

We have to keep in mind that this notebook does not try to train the best model, but to demosntrate how to perform parameter tuning in a professional way (including frameork for tracking & deployment). 

Now grab a dataset, and try this on real world data!

# Part 6 - Choosing the right model for the right question (~1h30)

So far you have trained three models on the same severity target (`gamma_model`, `xgb_model`, `final_model` from Optuna) and compared them with RMSE/MAE. In **M03_F06 (Mesures de performances)** you already learned a lot more about *how* to measure a model properly: choosing the right statistical metric for the problem (Poisson deviance vs. MSE...), building asymmetric business-cost scorers, and checking calibration (reliability plots, Brier score, Platt/isotonic recalibration). We will not repeat any of that here.

Instead, this part focuses on two questions M03_F06 does not cover, but that are just as decisive when picking "the best model" in practice:

1. **Can you actually afford to train and serve it?** (computational / operational cost)
2. **Will it still be good tomorrow?** (robustness over time — this directly resolves the cliffhanger from Part 5!)

We will finish with a small capstone where you combine *everything* (M03_F06's metrics + these two new criteria) to recommend a model to three stakeholders who don't ask the same question.


## 6.1 Computational and operational cost (~30 min)

A model that scores best on RMSE is not automatically the model you should ship. Two very different constraints show up constantly in insurance:

- **Batch repricing**: recompute a premium for the entire portfolio (millions of policies) overnight — training time and batch prediction throughput matter.
- **Real-time quoting**: a customer requests a quote online and expects an answer well under a second — single-row inference latency matters, and it usually matters *more* than a small accuracy gain.

Neither of these constraints is visible in an RMSE number. You have to measure them explicitly.

### 📝 Exercise 10 - Measure the cost of each model

Instructions:
- For `gamma_model`, `xgb_model` and `final_model` (the Optuna-tuned model), measure:
  - **Fit time**: how long does `.fit(...)` take on the training set?
  - **Predict time (batch)**: how long does `.predict(...)` take on the full test set?
  - **Predict time (single row)**: how long does `.predict(...)` take on *one* row only (simulate a real-time quote)?
- Put everything in a small `pandas.DataFrame` with one row per model and columns for fit time, batch predict time, single-row predict time, and the RMSE/MAE you already computed earlier in the notebook.


In [ ]:
# TODO: measure fit time, batch predict time and single-row predict time for gamma_model, xgb_model and final_model


#### 💡 Hints
- Use `time.perf_counter()` before/after the call you want to measure (avoid `%timeit` for the single-row case — it is convenient but re-runs the cell many times and hides the *first-call* overhead, which matters for cold-start serving).
- For the "single row" case, use `X_test.iloc[[0]]` (double brackets keep it a DataFrame with 1 row) so the model receives the same shape it expects in production.
- Don't forget that `gamma_model` needs the *transformed* (one-hot encoded) features (`X_train_transformed`/`X_test_transformed`), while `xgb_model`/`final_model` take the raw `X_train`/`X_test` with categorical dtypes.
- A single timing run is noisy — run each measurement 5-10 times and report the median, not a single value.

#### 🤔 Take a step back and discuss the results
- Which model is fastest to (re)train? Which is fastest to serve a single quote?
- Is the accuracy gain from `final_model` over `gamma_model` big enough to justify its extra cost, given the two scenarios above (batch repricing vs. real-time quoting)?
- A model that takes 2 hours to retrain is not a problem if you retrain it once a quarter. It *is* a problem if your MLOps pipeline expects a nightly retrain. Operational cost is only meaningful once you know the constraint it has to fit into.


## 6.2 Robustness over time (~30 min)

☠️ Remember the ending of Part 5: **the model performed very poorly on `claims_2024`**, because the train/test split back in Part 2 was a random split, not a time-based one — so the test set silently contained rows from the same years as the training set, and gave us an overly optimistic RMSE.

A model that looks great on a random split can still fail the moment it meets next year's data, if the world has changed (inflation, a new storm season, a new distribution of risks in the portfolio...). "Which model generalizes over time" is a criterion in its own right, separate from "which model fits best on average" — and M03_F06 does not cover it.

### 📝 Exercise 11 - Redo the split properly, and compare

Instructions:
- Rebuild `X_train`/`X_test`/`y_train`/`y_test` using a **time-based split** instead of the random one from Part 2: train on claims with `DATE_SURVENANCE` before 2024, test on `claims_2024`.
- Retrain a Gamma GLM and an XGBoost model (you can reuse the same hyperparameters as `gamma_model`/`final_model`) on this new split.
- Compare the RMSE/MAE you get here to the RMSE/MAE you got earlier in the notebook (random split).
- Also compute the **mean bias** (average prediction minus average actual value) on both splits. A per-row error metric and an aggregate bias can tell very different stories — don't assume one implies the other.



In [ ]:
# TODO: rebuild a time-based train/test split (train < 2024, test = 2024), retrain, and compare metrics to the random-split results from earlier


#### 💡 Hints
- You already built `claims_2019_2023` and `claims_2024` right at the very start of the notebook (Exercise 1) — reuse them instead of `train_test_split`.
- Apply the *same* feature engineering (`AGE`, `DEPARTEMENT`, `COUT_UNITAIRE`, `HAS_EXTENSION`) to both, and reuse the already-fitted `column_transformer` for the GLM (fit only on the training years, never on 2024).
- Keep the same hyperparameters you already tuned — the point here is *not* to re-optimize, it's to isolate the effect of the split method alone.
- Don't only look at RMSE/MAE: also compute the mean bias (`y_pred.mean() - y_true.mean()`) on both splits. A model can have a similar (or even better!) per-row error on the time-based split while still being badly biased in aggregate.

#### 🤔 Take a step back and discuss the results
- Does RMSE/MAE actually degrade on the time-based split, or does something else break instead? Look specifically at the mean bias for both models — is it larger on the time-based split, and is it similar in size for the GLM and for XGBoost?
- If you had shipped based on the random-split numbers alone, what would have gone wrong in production? Think in terms of over/under-pricing on average, not just "the error is bigger".
- This connects directly to the "Evaluate differently" idea from `docs/00_finding_the_best_model.md`: when data is time-sensitive, a random train/test split can hide this kind of aggregate bias — a single error metric (RMSE/MAE) is not always enough on its own.



## 6.3 Capstone - pick the right model for the right question (~30 min)

You now have, for the same severity problem, several candidate models (`gamma_model`, `xgb_model`, `final_model`) and several *independent* criteria to judge them on: statistical fit (RMSE/MAE, and everything M03_F06 taught you about metrics/calibration), computational/operational cost (Exercise 10), and robustness over time (Exercise 11).

"The best model" does not exist in the abstract — it depends on who is asking and why. Here are three stakeholders who will all look at your work and ask a different question:

| Persona | Their question | What they care about most |
|---|---|---|
| **Pricing committee** | "Can we defend this model to justify a premium change?" | Interpretability, calibration, being able to explain a coefficient/SHAP value in a meeting |
| **Real-time quoting API** | "Will this answer in time, at scale?" | Single-row inference latency, throughput, infrastructure cost |
| **Regulator / auditor** | "Will this still be valid next year?" | Robustness over time, documented validation methodology (time-based split!), explainability |

### 📝 Exercise 12 - Recommend a model to each persona

Instructions:
- Build a single comparison table (one row per model, one column per criterion: RMSE/MAE, fit time, single-row predict time, RMSE/MAE **and mean bias** on the time-based split, and a qualitative interpretability rating) using everything you measured in this notebook.
- For each of the three personas above, write 2-3 sentences recommending one model (it does not have to be the same model for all three!) and justifying the choice using the table.



In [ ]:
# TODO: assemble one comparison table (models x criteria) from your results in Exercises 3/4/6/10/11


#### ✍️ Your recommendation (fill this in)

- **Pricing committee**: ...
- **Real-time quoting API**: ...
- **Regulator / auditor**: ...

#### 💡 Hints
- There is no single correct answer here — what matters is that your recommendation is consistent with the numbers in your table, not with a gut feeling.
- It's fine (and realistic!) to recommend the same model for two personas and a different one for the third.
- If two models are close on every criterion that matters to a persona, say so explicitly instead of picking arbitrarily — "either would work, X is marginally better on Y" is a perfectly good answer.

---

You've now seen the full picture: M03_F06 gave you the vocabulary to *measure* a model correctly (the right metric, cost-weighted scoring, calibration); this part gave you two more axes — *can you afford to run it* and *will it still hold up over time* — and the discipline of matching your final choice to the actual question being asked, not to a single leaderboard number.
